# Training, Tuning and Evaluating LLMs


Lecture 12 covered the engineering of a large pretraining run: GPUs, parallelism,
scaling laws and data curation. This lecture follows the model from the end of
pretraining to the point where you can trust it on a task: the data it was trained
on, the three kinds of fine-tuning that turn a base model into a classifier or an
assistant, how to *measure* whether any of that worked, how to add knowledge without
retraining (RAG), and how to fine-tune cheaply (LoRA). The evaluation section is the
one you will use first: the final project asks you to build an eval before you build
a model.

UDL does not cover post-training or evaluation; where notation is needed we use that
of the primary paper and say so. Many of the figures in the fine-tuning sections are
from Sebastian Raschka's talk
*[Developing an LLM: Building, Training, Finetuning](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*
and his *[LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch)* repository;
other sources are cited where they appear.


## Topics

1. Pre-training data: what goes in, and how the targets are built
2. Classifier fine-tuning
3. Instruction (chat) fine-tuning
4. Preference tuning: what a reward model is (lecture 14 derives the rest)
5. Evaluating LLMs: how to build an eval you can trust
6. Retrieval-augmented generation and context engineering
7. Parameter-efficient fine-tuning: LoRA
8. Summary


## LLM Generative Flow

![](figs/13-Training-and-Evaluating-LLMs/llm-generative-flow.png)

- How do we improve the response?
- How do we evaluate the response?


## How do we build a chat model?

![*[State of GPT, Andrej Karpathy, MS Build Keynote](https://youtu.be/bZQun8Y4L2A?si=Sv66kPGvIVgNQKDY)*, 2023](figs/13-Training-and-Evaluating-LLMs/gpt-assistant-training-pipeline.png)

The four stages are still the map in 2026: pretraining on the web, supervised
fine-tuning on demonstrations, a reward model from comparisons, and reinforcement
learning against that reward. This lecture covers the first two stages and the
*idea* of the third; lecture 14 derives the third and fourth.


# Pre-training

The first stage turns an architecture into a foundation model by pretraining it on
unlabeled text. Lecture 12 covered how a run is engineered and how its budget is
split between parameters and tokens; here we look at what the data is, how much of
it there is, and how the training targets are built from it.


## Pre-Training

![*[LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch)*, Sebastian Raschka](figs/13-Training-and-Evaluating-LLMs/llm-build-stages-pretraining.png)

## The GPT-3 dataset was 499 billion tokens

| Dataset | Quantity (tokens) | Weight in Training Mix | Epochs Elapsed when Training for 300B Tokens |
|---|---|---|---|
| Common Crawl (filtered) | 410 billion | 60% | 0.44 |
| WebText2 | 19 billion | 22% | 2.9 |
| Books1 | 12 billion | 8% | 1.9 |
| Books2 | 55 billion | 8% | 0.43 |
| Wikipedia | 3 billion | 3% | 3.4 |


| Quantity (Tokens) | Percentage |
|---:|---:|
| 410 | 82% |
| 19 | 4% |
| 12 | 2% |
| 55 | 11% |
| 3 | 1% |
| **499** | |


Note that the mix weights are not the raw proportions: the small, high-quality sources
are up-sampled (Wikipedia is seen 3.4 times) and Common Crawl is down-sampled.

*Language Models are Few-Shot Learners* (Brown et al., 2020), <https://arxiv.org/abs/2005.14165>

*Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*

## Llama 1 was trained on 1.4T tokens

| Dataset | Sampling prop. | Epochs | Disk size |
|---|---:|---:|---:|
| CommonCrawl | 67.0% | 1.10 | 3.3 TB |
| C4 | 15.0% | 1.06 | 783 GB |
| Github | 4.5% | 0.64 | 328 GB |
| Wikipedia | 4.5% | 2.45 | 83 GB |
| Books | 4.5% | 2.23 | 85 GB |
| ArXiv | 2.5% | 1.06 | 92 GB |
| StackExchange | 2.0% | 1.03 | 78 GB |

Table 1: **Pre-training data.** Data mixtures used for pre-training, for each subset we
list the sampling proportion, number of epochs performed on the subset when training on
1.4T tokens, and disk size. The pre-training runs on 1T tokens have the same sampling
proportion.

*LLaMA: Open and Efficient Foundation Language Models* (Touvron et al., 2023), <https://arxiv.org/abs/2302.13971>

*Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*

## Llama 2: 2T tokens. Llama 3: 15T tokens.

> "Our training corpus includes [a new mix of data from publicly available sources]{.mark},
> which does not include data from Meta's products or services. [...] We trained on
> [2 trillion tokens]{.mark} of data as this provides a good performance–cost trade-off,
> up-sampling the most factual sources in an effort to increase knowledge and dampen
> hallucinations."
>
> *Llama 2: Open Foundation and Fine-Tuned Chat Models* (Touvron et al., 2023), <https://arxiv.org/abs/2307.09288>

> "Llama 3 is pretrained on over [15T tokens that were all collected from publicly
> available sources.]{.mark}"
>
> *Introducing Meta Llama 3* (2024), <https://ai.meta.com/blog/meta-llama-3/>; details in
> [*The Llama 3 Herd of Models*](https://arxiv.org/abs/2407.21783), section 3.1

**Why did they stop listing training sources?** Llama 3's report describes the
*pipeline* (extraction, deduplication, quality classifiers, mixing) in detail but not
the sources. Lecture 12, section "Data at scale", walks through that pipeline.


*Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*

## Quantity vs quality

> "we mainly focus on the [quality of data]{.mark} for a given scale. We try to calibrate
> the training data to be closer to the "data optimal" regime for small models. In
> particular, we filter the publicly available web data to contain the correct level of
> "knowledge" and keep more web pages that could potentially improve the "reasoning
> ability" for the model. As an example, [the result of a game in premier league in a
> particular day might be good training data for frontier models, but we need to remove
> such information to leave more model capacity for "reasoning"]{.mark} for the mini size
> models.

*Phi-3 Technical Report: A Highly Capable Language Model Locally on Your Phone* (2024),
<https://arxiv.org/abs/2404.14219>

How many tokens a model of a given size *should* see, and how the budget splits between
$N$ and $D$, is the scaling-law question: see lecture 12, "Scaling laws".

*Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*

## Labels are the inputs shifted by +1

![*Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*](figs/13-Training-and-Evaluating-LLMs/labels-shifted-inputs.png)

There is no labeling step: the loss is the cross-entropy of the next token, so every
position in every document is a training example and the target tensor is the input
tensor shifted by one. This is why "unlabeled" text is enough, and why the data
*is* the supervision: whatever is in the corpus is what the model learns to predict.


## Training for ~1–2 epochs is usually a good sweet spot

![*Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*](figs/13-Training-and-Evaluating-LLMs/pretraining-loss-epochs.png)

A language model memorizes its corpus quickly: after one or two passes the training
loss keeps falling while the validation loss stops. That is why the token-count
tables above show most sources seen about once, and why "more unique tokens" beats
"more epochs". Lecture 12's "Reading a loss curve" slide shows the same split on a
real run.


# Classifier Fine-Tuning

With a pretrained foundation model in hand, the simplest adaptation is to turn it into
a classifier: swap the output layer for one sized to the classes and finetune on
labeled examples. Everything from lectures 3 to 6 applies unchanged.


## Classifier Finetuning

![*Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*](figs/13-Training-and-Evaluating-LLMs/stage3-classifier-finetuning.png)

## Replace the output layer: a spam/ham classifier

![*Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*](figs/13-Training-and-Evaluating-LLMs/spam-classifier-output-layer.png)

The pretrained linear output layer maps the 768-dimensional hidden state to 50,257
vocabulary logits. For a two-class task it is replaced by a freshly initialized
$768 \times 2$ layer, read at the last token, and trained with the usual cross-entropy.


## Track loss, and task performance

![](figs/13-Training-and-Evaluating-LLMs/classifier-loss-curves.png)


![](figs/13-Training-and-Evaluating-LLMs/classifier-accuracy-curves.png)


Loss is what you optimize; accuracy (or whatever the task metric is) is what you
report. Watch both: the loss curves look fine well past the point where training
accuracy hits 1.0.

*Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*

## We don't need to finetune all layers

![<https://magazine.sebastianraschka.com/p/finetuning-large-language-models>. *Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*](figs/13-Training-and-Evaluating-LLMs/finetune-layers-training-time.png)

Training only the last layer is fast but weak; training the last two transformer
blocks plus the head reaches the accuracy of full fine-tuning at a fraction of the
time. This is the first hint that most of the adaptation lives in a small part of the
network, which the LoRA section turns into a method.


# Instruction (Chat) Fine-Tuning

The other branch of Stage 3 finetunes the foundation model on instruction–response
pairs, turning it into an assistant. The loss is the same next-token cross-entropy as
pretraining; what changes is the data and the template that wraps it.


## Instruction finetuning

![*Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*](figs/13-Training-and-Evaluating-LLMs/stage3-instruction-finetuning.png)

## Applying a prompt template

```json
{
    "instruction": "Rewrite the following sentence using passive voice.",
    "input": "The team achieved great results.",
    "output": "Great results were achieved by the team."
},
```

↓ **Apply prompt style template (for example, Alpaca-style)**

```text
Below is an instruction that describes a task. Write a response
that appropriately completes the request.

### Instruction:
Rewrite the following sentence using passive voice.

### Input:
The team achieved great results.

### Response:
Great results were achieved by the team.
```

↓ **Pass to LLM for supervised instruction finetuning**

*Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*

## Model input and model response

**Model input**

```text
Below is an instruction that describes a task. Write a response
that appropriately completes the request.

### Instruction:
Rewrite the following sentence using passive voice.

### Input:
The team achieved great results.
```

**Model response**

```text
### Response:
Great results were achieved by the team.
```

The whole string is tokenized and trained with next-token loss. Most recipes mask the
loss on the prompt tokens so the model is only trained to produce the response.
Modern chat templates (Llama 3, Qwen, Gemma) do the same job with special tokens
marking the system, user and assistant turns.

*Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*

## Alpaca Instruction Tuning Dataset

![The `tatsu-lab/alpaca` dataset on Hugging Face: 52k rows of instruction, input and output, generated by prompting `text-davinci-003` ([Taori et al., 2023](https://crfm.stanford.edu/2023/03/13/alpaca.html)).](figs/13-Training-and-Evaluating-LLMs/alpaca-dataset-hf.png)

## LIMA: Finetuning with only 1K instructions

![The LIMA paper ([Zhou et al., 2023, arXiv:2305.11206](https://arxiv.org/abs/2305.11206)) and the `GAIR/lima` dataset on Hugging Face: 1.03k rows.](figs/13-Training-and-Evaluating-LLMs/lima-paper-and-dataset.png)

LIMA's claim, the *superficial alignment hypothesis*, is that almost all of a model's
knowledge is learned in pretraining and instruction tuning only teaches the format
and style of an answer, so a thousand carefully written examples can be enough. The
claim was tested, of course, by human preference evaluation against other models,
which is the subject of the next two sections.


# Preference Tuning

Instruction tuning teaches a model to follow a task; preference tuning then shapes
which of several valid responses it prefers to give. This section introduces the
idea and the one object every method shares, the reward model. Lecture 14 derives
the training algorithms (RLHF with PPO, DPO, GRPO).


## Refine responses for style or safety: reward preferred responses

**Input Prompt:**\
"What are the key features to look for when purchasing a new laptop?"

**Answer 1: Technical Response**

"When purchasing a new laptop, focus on key specifications such as the processor speed,
RAM size, storage type (SSD vs. HDD), and battery life. The processor should be powerful
enough for your software needs, and sufficient RAM will ensure smooth multitasking. Opt
for an SSD for faster boot times and file access. Additionally, screen resolution and
port types are important for connectivity and display quality."


**Answer 2: User-Friendly Response**

"When looking for a new laptop, think about how it fits into your daily life. Choose a
lightweight model if you travel frequently, and consider a laptop with a comfortable
keyboard and a responsive touchpad. Battery life is crucial if you're often on the move,
so look for a model that can last a full day on a single charge. Also, make sure it has
enough USB ports and possibly an HDMI port to connect with other devices easily."


Both are correct. Which one a model should give is a matter of *preference*, and
there is no gold "output" to put in a supervised dataset. Instead a human says which
of the two they prefer.

*Source: [S. Raschka, "Developing an LLM: Building, Training, Finetuning"](https://youtu.be/kPGTx4wcm_w?si=2sFKkDLmshHd5xaK)*

## What a reward model is

Notation of [Ouyang et al. (2022), InstructGPT](https://arxiv.org/abs/2203.02155).

- Collect comparisons: for prompt $x$, two responses, a human marks the winner $y_w$
  and the loser $y_l$.
- Train a **reward model** $r_\theta(x, y) \in \mathbb{R}$ (an LLM with its output
  head replaced by a single scalar, exactly like the classifier head above) so that
  the preferred response scores higher:

$$
\mathrm{Pr}(y_w \succ y_l \mid x) = \sigma\big(r_\theta(x, y_w) - r_\theta(x, y_l)\big),
\qquad
L = -\log \sigma\big(r_\theta(x, y_w) - r_\theta(x, y_l)\big)
$$

- This is the **Bradley–Terry model** of pairwise comparisons: only score
  *differences* matter, and a gap of $\Delta$ in reward gives win probability
  $\sigma(\Delta)$. A gap of 2 means the winner is preferred with probability
  $\sigma(2) \approx 0.88$.
- Then adjust the policy to produce responses with high $r_\theta$ while staying close
  to the SFT model. *How* (PPO, DPO, GRPO) is lecture 14; what matters here is that
  the training signal is now **a judgement of whole responses**, not next-token
  likelihood.
- The same Bradley–Terry model scores models on Chatbot Arena, where the "reward
  model" is the crowd: see the evaluation section.


## Why can validation loss improve and human judgement get worse?

In-class question: *You fine-tune an LLM using next-token prediction loss. The
validation loss improves, but human judges rate the outputs as worse. (1) Why can this
happen (list at least two reasons)? (2) Propose one change to training and one change
to evaluation.*


Validation loss is only a proxy objective, while humans care about behavior, intent, and
quality of whole responses.

More specifically:

1. Objective mismatch
   - Validation loss: how well does the model predict next token?
   - Human eval: "Is this helpful, correct, clear, safe and aligned with intent?"
2. Overfitting surface patterns rather than substance
   - May use common phrases, boilerplate answers, etc., when humans might penalize
     generic answers, overly verbose, etc.
3. Reward hacking of likelihood
   - Model learns shortcuts that optimize loss but not usefulness

## Propose one change to training and one change to evaluation

Change to **training** (any one of these)

1. Switch from pure next-token loss to **instruction tuning**: fine-tune on
   *(instruction, response)* pairs. *Optimize for task-following, not token prediction.*
2. Add **preference-based training**: train a reward model from human comparisons and
   optimize for preferred responses. *Use human judgement as the training signal.*
3. **Mix losses**: likelihood plus a KL penalty to a reference model, or penalties for
   verbosity, repetition, unsafe content. *Next-token loss is an incomplete objective.*
4. **Early stopping**: validation loss can keep improving while behaviour degrades.
   *Better likelihood ≠ better behaviour.*


Change to **evaluation** (any one of these)

1. Add **human evaluation**: pairwise preferences or task-specific rubrics
   (helpfulness, correctness, tone). *Evaluate what you actually care about.*
2. Use **task-aligned metrics**: exact match / F1 (QA), pass@$k$ (code),
   instruction-following score, hallucination rate. *Likelihood is a proxy, not the goal.*
3. Evaluate **multiple generations** per prompt and aggregate. *Single deterministic
   outputs hide failure modes.*
4. Do **targeted error analysis**: hallucination, instruction non-compliance, reasoning
   errors, toxicity. *Diagnose behaviour, not average loss.*


The rest of this lecture's evaluation section is how to do the right-hand column well.

# Evaluating LLMs

Since likelihood is only a proxy, we need other ways to judge a generative model.
Rather than tour the leaderboards (they change monthly; a web note below says where
to look), this section is about how an evaluation is *built*: what a benchmark
measures, why held-out logic is harder for LLMs than for a classifier, how LLM
judges fail, what crowd-preference arenas do and do not tell you, and how to build an
eval of your own for the final project.


## What are we evaluating?

A generative model has no single "accuracy". Depending on the task you may care about

- **Correctness**: is it factual, or hallucinated? Does the code run?
- **Relevance**: does it answer the question that was asked?
- **Safety and bias**: is it fair? Toxic? Does it refuse what it should?
- **Style and format**: length, tone, following the requested schema
- **Consistency**: does it give the same answer twice? Should it?

Three sources of a score, each with its own failure modes:

1. **A benchmark** with reference answers (MMLU, GSM8K, SWE-bench, ...)
2. **A human** (expert graders, or the crowd on an arena)
3. **An LLM judge** standing in for the human


## What a benchmark measures

A benchmark is three things: a **task definition**, a **set of items** with reference
answers, and a **scoring rule**. The number it reports is only as meaningful as the
weakest of the three.

| Benchmark | Task | Scoring |
|---|---|---|
| [MMLU](https://arxiv.org/abs/2009.03300) | 57-subject multiple choice | exact match (A–D) |
| [GSM8K](https://arxiv.org/abs/2110.14168) | grade-school word problems | final number |
| [HellaSwag](https://arxiv.org/abs/1905.07830) | sentence completion | pick the ending |
| [TruthfulQA](https://arxiv.org/abs/2109.07958) | questions with common misconceptions | judge or MC |
| [HumanEval](https://arxiv.org/abs/2107.03374) | write a function | pass@$k$ on unit tests |


Questions to ask of any number:

- **Prompt format and shots.** 0-shot vs. 5-shot, chain-of-thought or not, change
  scores by many points.
- **What counts as correct.** "Final number" scoring ignores the reasoning; a judge
  can be fooled by it.
- **Saturation.** When the frontier is above 90% the benchmark no longer separates
  models (MMLU in 2024).
- **Is it held out?** The next slide.


Lecture A4 covered train/validation/test discipline for a classifier: the test set is
touched once and the model never sees it. For an LLM, "the model never saw it" has to
hold against a 15-trillion-token corpus scraped from the same web where benchmarks
are published and discussed, which is the contamination problem.


## Held-out logic and contamination

A test score estimates generalization only if the test items were **not in the
training set**. For an LLM the training set is the public web, so published benchmarks
leak into it: as answer keys on GitHub, as worked solutions on forums, as copies in
other datasets.

**Decontamination, as done for Llama 3**
([Grattafiori et al., 2024](https://arxiv.org/abs/2407.21783), section 5.1.4)

- Measure **8-gram overlap** between each benchmark item and the pretraining corpus;
  an item is contaminated if a large enough fraction of its tokens lies in 8-grams
  that also occur in training data.
- Report the score on the clean subset next to the full score; the gap is the
  estimated gain from contamination.
- Why 8-grams and not exact match? Paraphrase and reformatting are the common
  case; a threshold on overlapping $n$-grams catches more of it, at the cost of
  false positives on boilerplate.
- [Singh et al. (2024)](https://arxiv.org/abs/2411.03923) compare these detection
  choices across benchmarks and find that the threshold matters as much as the
  method.


**Does it matter?** [GSM1k](https://arxiv.org/abs/2405.00332) (Zhang et al., 2024)
rewrote 1,250 new problems matching GSM8K's difficulty and style and never
published them. Some model families scored up to about 8 points lower on GSM1k
than on GSM8K, while others showed no gap.

**Defences**

- Keep a **private held-out split** (HLE, ARC-AGI do this).
- **Refresh** items on a schedule (LiveBench, LiveCodeBench).
- Prefer items **generated after** the model's data cutoff.
- For your own eval: never paste your test items into a chat with the model
  you are testing.


## LLM-as-judge

Human grading is the gold standard and does not scale. [Zheng et al. (2023)](https://arxiv.org/abs/2306.05685)
(*Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*) asked whether a strong LLM
can grade instead.

**The finding.** On MT-Bench, GPT-4's verdicts agreed with human experts about as
often as the experts agreed with each other (over 80%). A strong judge is a usable
proxy for a human, *on average*.

**The biases they documented**

- **Position bias**: the judge favours the first (or second) answer shown.
  [Wang et al. (2023)](https://arxiv.org/abs/2305.17926) showed rankings can flip
  just by swapping the order.
- **Verbosity bias**: longer answers win, even when the extra text adds nothing.
- **Self-enhancement bias**: a judge prefers text in its own style.
  [Panickssery et al. (2024)](https://arxiv.org/abs/2404.13076) tie this to the
  judge *recognizing* its own output.
- **Weak at math and reasoning**: it can be led astray by a confident wrong answer.


**Later work**

[Ye et al. (2024)](https://arxiv.org/abs/2410.02736) catalogue twelve judge biases
(position, verbosity, self-preference, authority, bandwagon, sentiment, ...) and
measure each by perturbing the answers and watching the verdict move.

**Mitigations that are now standard practice**

- Grade each pair **twice with the order swapped**; count a win only if consistent.
- Give the judge a **reference answer** or a **rubric**, not just "which is better".
- Ask for **reasoning before the verdict** (chain-of-thought judging).
- Use a **different model family** as judge than the one being evaluated.
- **Check the judge against humans** on a sample before trusting it (how: two
  slides on).


## Pairwise vs. rubric grading

**Pairwise (relative)**

"Here are responses A and B to the same prompt. Which is better?"

- Easier judgement; humans and LLMs are more consistent at *comparing* than at
  *scoring*.
- Produces win rates, and with many pairs a Bradley–Terry ranking.
- Needs a baseline to compare against (AlpacaEval and Arena-Hard compare every
  model to a fixed reference model).
- Cannot say whether *either* answer is acceptable.


**Single-answer rubric (absolute)**

"Score this response 1–10 for correctness, completeness and tone", or a checklist of
pass/fail criteria.

- Gives an absolute number per model and per criterion, which is what a product
  decision needs ("is it good enough?").
- Scores drift between judges, prompts and days; needs anchoring with a reference
  answer and example scores.
- Checklists of yes/no questions ("does it cite the source?") are far more reliable
  than a 1–10 scale.


Practical rule: use **pairwise** to choose between two systems, a **binary rubric**
to track one system over time, and never a 1–10 scale without anchors.

## Arena-style preference evals

![Arena battle mode: two anonymous models answer the same prompt, the user votes, and model identities are revealed afterwards. <https://arena.ai>](figs/13-Training-and-Evaluating-LLMs/arena-battle-mode.png)

[Chatbot Arena](https://arxiv.org/abs/2403.04132) (Chiang et al., 2024): crowdsourced,
pairwise, blind, on the users' own prompts. Each vote is a comparison; the leaderboard
is a Bradley–Terry fit to all of them.

## Arena scores are Bradley–Terry ratings

In [ ]:
#| code-fold: true
#| fig-cap: "Win probability against score gap under the Elo-scaled Bradley–Terry model used by Chatbot Arena."
#| fig-alt: "S-shaped curve of win probability against Arena score gap from minus 600 to plus 600. Marked points: a gap of 0 gives 50%, 100 gives 64%, 200 gives 76%, 400 gives 91%."
import numpy as np
import matplotlib.pyplot as plt

scale = 400                    # Elo scale: a gap of `scale` points is 10:1 odds
marked_gaps = [0, 100, 200, 400]

def win_prob(gap):
    return 1 / (1 + 10 ** (-gap / scale))

gap = np.linspace(-600, 600, 601)
fig, ax = plt.subplots(figsize=(6.4, 3.8), dpi=76)
ax.plot(gap, win_prob(gap), color="#2a78d6", lw=2.2)
for g in marked_gaps:
    ax.plot(g, win_prob(g), "o", color="#eb6834", ms=6, zorder=3)
    ax.annotate(f"{g:+d} pts: {100 * win_prob(g):.0f}%", (g, win_prob(g)),
                xytext=(-78, 6) if g == 0 else (10, -16), textcoords="offset points",
                fontsize=10)
ax.axhline(0.5, color="#8a8984", lw=0.8, ls="--")
ax.set_xlabel("Arena score gap  $s_i - s_j$")
ax.set_ylabel("P(model $i$ wins the battle)")
ax.set_xlim(-600, 600)
ax.set_ylim(0, 1)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.grid(color="#dcdcd8", lw=0.6)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

Each model $i$ gets a score $s_i$; the model says

$$
\mathrm{Pr}(i \succ j) = \frac{1}{1 + 10^{-(s_i - s_j)/400}}
$$

and the scores are fit by maximum likelihood to the votes (a logistic regression
on indicator features, one per model).

- Same model as the reward-model loss, on a different scale: $10^{\Delta/400}$ in place
  of $e^{\Delta}$.
- **Worked example.** Model A at 1500, model B at 1400: A wins a battle with
  probability $1/(1 + 10^{-100/400}) = 1/(1 + 0.562) \approx 0.64$. A 100-point
  lead is a 64/36 split, not a landslide.
- Scores come with confidence intervals; two models whose intervals overlap are tied,
  whatever the rank column says.


The same formula as a calculator, in both directions: from a score gap to a win
rate, and from a win rate back to the gap it implies,
$\Delta = 400 \log_{10}\!\big(p/(1-p)\big)$.

In [ ]:
s_a, s_b = 1500, 1400          # Arena scores of models A and B
scale = 400

gap = s_a - s_b
p = 1 / (1 + 10 ** (-gap / scale))
print(f"gap {gap:+d}: A wins with probability {p:.3f} (odds {p / (1 - p):.2f} : 1)")

for g in [50, 100, 200, 400, 800]:
    print(f"gap {g:>4d}: P(win) = {1 / (1 + 10 ** (-g / scale)):.3f}")

for target in [0.55, 0.60, 0.75, 0.90]:
    print(f"P(win) = {target:.2f} needs a gap of {scale * np.log10(target / (1 - target)):.0f} points")

## What an arena can and cannot tell you

**Strengths**

- Real prompts from real users, not a fixed test set, so it is hard to contaminate.
- Measures what people *prefer*, the thing preference tuning optimizes.
- Blind, so brand effects are removed (at least until the model names itself).
- Hundreds of thousands of votes: tight intervals at the top.

**Built-in limits**

- Preference is not correctness: a confident, well-formatted wrong answer can win.
- Inherits the judge biases above: votes favour longer and more decorated answers
  (Arena now reports a **style-controlled** score for this reason).
- The prompt distribution is whoever shows up: short, English, chat-like.


**The Leaderboard Illusion** ([Singh et al., 2025](https://arxiv.org/abs/2504.20879))

- **Private testing with selective disclosure**: providers can test many variants
  anonymously and publish only the best. The paper identifies 27 private variants
  tested by one provider before a single launch. Picking the max of $n$ noisy scores
  biases the result upward.
- **Unequal data access**: a few proprietary providers receive a large share of all
  battle data, and the authors show that training on Arena data raises Arena
  performance.
- **Silent deprecation**: many open-weight models are removed from sampling, so the
  comparison graph becomes sparse and the Bradley–Terry fit less reliable for them.

The lesson is general: **any leaderboard that lets participants choose what to
submit is an optimization target, and Goodhart's law applies.**


## Task-specific evals, and their design logic

| Eval | What an item is | Why it is built that way |
|---|---|---|
| [**SWE-bench**](https://arxiv.org/abs/2310.06770) (Jimenez et al., 2024) | A real GitHub issue from a Python repo; the model must produce a patch; scored by the repo's own **hidden unit tests** | Outcome is *verifiable* and *end-to-end*; no judge needed. The **Verified** subset (500 items) was human-screened because many original issues were under-specified. |
| [**GPQA**](https://arxiv.org/abs/2311.12022) (Rein et al., 2023) | 448 multiple-choice questions written by PhD-level experts in biology, physics and chemistry | *Google-proof*: skilled non-experts with unrestricted web access scored about 34%; experts about 65%. Tests knowledge that cannot be looked up. |
| [**Humanity's Last Exam**](https://arxiv.org/abs/2501.14249) (Phan et al., 2025) | 2,500 closed-ended questions from about 1,000 expert contributors, filtered so frontier models failed them at submission time | *Adversarially* collected at the frontier of human knowledge; a **private held-out split** detects overfitting to the public set. |
| [**ARC-AGI**](https://arxiv.org/abs/1911.01547) (Chollet, 2019; [ARC-AGI-2](https://arxiv.org/abs/2505.11831), 2025) | A few input–output grid pairs; infer the rule, apply it to a new grid | Each task is *novel*, so memorization does not help; easy for humans, hard for models. Scores are reported **against cost per task**, on a private test set. |

Four different answers to "how do we stop the score from being memorization?":
verifiable outcomes, un-googleable expertise, adversarial collection with a private
split, and novelty by construction.

## Benchmarks saturate; the frontier moves

![Accuracy of early-2025 models across benchmarks. Figure from [Phan et al. (2025)](https://arxiv.org/abs/2501.14249).](figs/13-Training-and-Evaluating-LLMs/hle-benchmark-accuracy.png)


![Two evaluations of the same models disagree in places: LiveBench (reference answers, refreshed questions) vs. Chatbot Arena (human preference), 2024. Figure from [livebench.ai](https://livebench.ai).](figs/13-Training-and-Evaluating-LLMs/livebench-vs-chatbot-arena.png)


Every benchmark in the left panel was "hard" when released. Within about two years
of release, MMLU, MATH and GPQA each went from near-chance to above 80% at the frontier,
and HLE scores rose from single digits at release to much higher within a year (check
the live leaderboard). A benchmark's useful life is the window
between "models can't" and "models have seen it".

**Where to look (this list goes stale; check the dates on whatever you open).**
Human preference: [arena.ai](https://arena.ai/leaderboard/) (text, code, vision,
search and image arenas, with style control). Reference-answer benchmarks with
refreshed items: [livebench.ai](https://livebench.ai),
[LiveCodeBench](https://livecodebench.github.io/). Aggregated indexes with speed and
price: [artificialanalysis.ai](https://artificialanalysis.ai/),
[epoch.ai/benchmarks](https://epoch.ai/benchmarks). Task-specific:
[swebench.com](https://www.swebench.com/), [lastexam.ai](https://lastexam.ai/),
[arcprize.org/leaderboard](https://arcprize.org/leaderboard),
[Scale SEAL](https://scale.com/leaderboard). Model cards and technical reports give
the numbers the provider chose to publish, with the prompt format in the appendix,
which is the first thing to check when two sources disagree.


## Building your own eval (final project)

The final project asks for an eval *before* a model. A usable one has five parts:

1. **A task definition** narrow enough to grade: one input type, one output type, one
   notion of "correct". "Summarizes well" is not a task; "extracts the five fields in
   this schema from a clinical note" is.
2. **Items from the real distribution.** 50 to 200 inputs sampled from where the
   system will be used, not written by you to be easy or hard. Include the ugly ones.
3. **A held-out split.** A *dev* set you may look at while iterating on prompts and
   models, and a *test* set you score at the end and never paste into a prompt.
   Tune on the test set and your number is a training score.
4. **A grader**, in order of preference: exact match or unit tests (verifiable);
   a binary rubric checked by an LLM judge (with order-swap and a reference answer);
   pairwise against a baseline. A 1–10 "quality" score is a last resort.
5. **An agreement check.** Hand-grade 50 items yourself and measure how often the
   grader agrees. If it does not agree with you, its score of the model means nothing.


## How good is the grader, and how sure is the score?

**Grader agreement.** Raw agreement $p_o$ is inflated when one label dominates;
report **Cohen's $\kappa$**:

$$
\kappa = \frac{p_o - p_e}{1 - p_e}
$$

where $p_e$ is the agreement expected by chance from each grader's label frequencies.

*Example.* Judge and human agree on 85 of 100 items; both mark "pass" 70% of the time.
$p_e = 0.7 \cdot 0.7 + 0.3 \cdot 0.3 = 0.58$, so
$\kappa = (0.85 - 0.58)/(1 - 0.58) \approx 0.64$: substantial, not perfect.
Below about 0.6, fix the rubric before scoring models.


**Uncertainty of a score.** A pass rate $\hat p$ on $n$ items has standard error

$$
\mathrm{SE} = \sqrt{\frac{\hat p (1 - \hat p)}{n}}
$$

*Example.* $\hat p = 0.80$ on $n = 100$ items: $\mathrm{SE} = \sqrt{0.16/100} = 0.04$,
so the 95% interval is about $80 \pm 8$ percent. Two systems at 80% and 85% on the
same 100 items are **not distinguishable**; to resolve a 5-point gap you need
several hundred items (or paired comparisons on the same inputs, which are more
sensitive).

Always report $n$ and an interval next to the number.


These two calculations are the quizzable content of this section: given agreement
counts, compute $\kappa$; given a pass rate and $n$, compute the standard error and
say whether a difference is resolved. The reasoning carries over to every
leaderboard you will read: a one-point gap on a 500-item benchmark is noise.

Both calculations, with the slide's numbers as inputs. The last part inverts the
standard error: a 95% half-width $h$ needs $n = \hat p(1-\hat p)(1.96/h)^2$
items, and to separate two independently scored systems the half-width applies to
their *difference*, whose variance is the sum of the two.

In [ ]:
# Cohen's kappa from agreement and each grader's "pass" rate
p_o = 0.85                     # fraction of items where judge and human agree
pass_judge, pass_human = 0.70, 0.70

p_e = pass_judge * pass_human + (1 - pass_judge) * (1 - pass_human)
kappa = (p_o - p_e) / (1 - p_e)
print(f"p_e = {p_e:.2f}, kappa = {kappa:.3f}")

# Standard error and 95% interval of a pass rate
p_hat, n = 0.80, 100
z = 1.96                       # 95% two-sided

se = np.sqrt(p_hat * (1 - p_hat) / n)
print(f"\np_hat = {p_hat:.2f} on n = {n}: SE = {se:.3f}, "
      f"95% interval {100 * p_hat:.0f} ± {100 * z * se:.1f} points")

# How many items for a target half-width?
for h in [0.08, 0.05, 0.03, 0.01]:
    print(f"±{100 * h:.0f} points needs n >= {np.ceil(p_hat * (1 - p_hat) * (z / h) ** 2):,.0f}")

# Are 80% and 85% distinguishable?
p1, p2 = 0.80, 0.85
se_diff = np.sqrt(p1 * (1 - p1) / n + p2 * (1 - p2) / n)
n_needed = (p1 * (1 - p1) + p2 * (1 - p2)) * (z / (p2 - p1)) ** 2
print(f"\n{100 * p1:.0f}% vs {100 * p2:.0f}% on n = {n} each: gap / SE = {(p2 - p1) / se_diff:.2f} "
      f"(need 1.96); resolving it takes n >= {np.ceil(n_needed):.0f} per system")

# Improving LLMs with RAG

If the eval says the model lacks knowledge your task needs, the cheapest fix is
usually not training. Retrieval-augmented generation improves responses by giving
the model relevant external information at query time. This section is a compact
introduction; lecture 21 returns to context engineering in the setting of agents.


## Retrieval-Augmented Generation (RAG)

RAG enhances LLMs by referencing external knowledge to generate relevant responses
([Lewis et al., 2020](https://arxiv.org/abs/2005.11401) coined the term).

- Integrates external data into LLM text generation.
- Reduces hallucination, improves response relevance, and lets you cite sources.
- Updates knowledge by changing the index, not the weights.
- Works with unstructured data (documents), structured data (SQL) and code.

![](figs/13-Training-and-Evaluating-LLMs/rag-overview.png)

## RAG – Loading and Indexing (offline)

![<https://docs.langchain.com/oss/python/langchain/rag>](figs/13-Training-and-Evaluating-LLMs/rag-loading-indexing.png)

A pipeline for ingesting data from a source and indexing it. Usually runs offline;
the choices that matter most are chunk size, chunk overlap and the embedding model.

## RAG – Retrieval and Generation (at query time)

![<https://docs.langchain.com/oss/python/langchain/rag>](figs/13-Training-and-Evaluating-LLMs/rag-retrieval-generation.png)

Takes the user query at run time, embeds it with the *same* embedding model, retrieves
the top-$k$ most similar chunks, and puts them in the prompt.

## RAG – Retrieval similarity measure

![<https://python.langchain.com/docs/modules/data_connection/vectorstores/>](figs/13-Training-and-Evaluating-LLMs/rag-vector-store.png)


"Most similar" means smallest **distance** $d$ between query embedding $\mathbf{A}$
and chunk embedding $\mathbf{B}$. The three offered by Chroma:

Squared L2 distance\*: $d = \sum_i (A_i - B_i)^2$

Inner-product distance: $d = 1 - \sum_i A_i B_i$

Cosine distance: $d = 1 - \dfrac{\sum_i A_i B_i}{\sqrt{\sum_i A_i^2}\,\sqrt{\sum_i B_i^2}}$

\* Default on the Chroma vector database. Cosine distance is one minus cosine
*similarity*; for unit-length embeddings all three rank chunks identically.

<https://docs.trychroma.com/usage-guide#changing-the-distance-function>



## Beyond a single dot product: bi-encoders, cross-encoders, hybrids

Is one similarity score between query and chunk the best we can do?

- **Bi-encoder (dense retrieval).** Query and passage are embedded *separately*, so
  passages can be indexed once and compared with a dot product.
  [DPR, Karpukhin et al. (2020)](https://arxiv.org/abs/2004.04906) trains the two
  encoders with a contrastive loss over (question, positive passage, hard negatives).
  Fast; the standard first stage.
- **Cross-encoder (reranking).** Query and passage are concatenated and read
  *together* by a transformer, so every query token attends to every passage token.
  [Nogueira and Cho (2019)](https://arxiv.org/abs/1901.04085) fine-tuned BERT this
  way; far more accurate, but it cannot be indexed, so it is used to rerank the
  top-$k$ from the first stage.
- **Hybrid.** Combine dense scores with lexical BM25 (exact terms, product codes,
  names), then rerank. Most production systems do this.
- **Late interaction.** [ColBERT, Khattab and Zaharia (2020)](https://arxiv.org/abs/2004.12832)
  keeps per-token embeddings and scores by max-similarity, a middle ground.

## Evaluating a RAG system: the RAG triad

![The RAG triad. <https://www.trulens.org/getting_started/core_concepts/rag_triad/>](figs/13-Training-and-Evaluating-LLMs/rag-triad.png)


Three questions, one per edge, each gradable by a rubric or a judge:

- **Context relevance** (query → context): is what we retrieved about the question?
  Irrelevant chunks get woven into the answer and read as hallucination.
- **Groundedness** (context → response): can every claim in the answer be supported
  by the retrieved context?
- **Answer relevance** (response → query): did the context pull the model away from
  the question that was asked?

Scoring the edges separately tells you *which stage* to fix: retrieval (chunking,
embedding, reranking) or generation (prompt, model). Everything from the evaluation
section applies: held-out questions, a binary rubric, an agreement check.


## Retrieval-Augmented Generation (RAG) paradigms

![Naive, Advanced and Modular RAG. Figure 3 of [Gao et al. (2023), *Retrieval-Augmented Generation for Large Language Models: A Survey*, arXiv:2312.10997](https://arxiv.org/abs/2312.10997).](figs/13-Training-and-Evaluating-LLMs/rag-paradigms.png)

**Pre-retrieval**: better chunks and metadata, query rewriting and expansion.
**Post-retrieval**: rerank, compress, fuse. **Modular**: routing, memory, iterative
retrieve-then-read. Each addition is a latency cost; measure it with the triad.

## From RAG to context engineering

RAG is one technique for adding external knowledge. **Context engineering** is the
broader discipline of managing *everything* the model sees before it generates:
retrieved chunks, tool outputs, memory, instructions, and the budget they share.

- **Orchestration**: which data, tools and memories a given step needs.
- **Precision filtering**: strip irrelevant chunks before they reach the model.
- **Routing**: a planner picks sources and tools (e.g. via the Model Context
  Protocol) before execution.
- **Memory**: distil facts from past turns into long-term memory; keep recent notes
  in context.
- **Contextual retrieval**: prepend a chunk-level summary at ingestion so a snippet
  carries its own context ([Anthropic, 2024](https://www.anthropic.com/news/contextual-retrieval)).

Lecture 21 picks this up where it matters most: agents that fill their own context.


![](figs/13-Training-and-Evaluating-LLMs/rag-to-context-engineering.png)


# Parameter Efficient Fine-Tuning: Low-Rank Adaptation

When prompting and retrieval are not enough, we fine-tune. Full fine-tuning is
expensive in compute and storage, which motivates updating only a small, low-rank
part of the model. LoRA is the tool you will most likely use in the final project,
so this section is kept in full. Notation follows Hu et al. (2021).


## Model Finetuning: when, and what it costs

- Large foundation models are pre-trained on general tasks and may not do well on
  specialized ones.
  - Try prompt engineering and retrieval augmentation first.
- Good news: a much smaller dataset adapts the model to a downstream task.
- Bad news: a fine-tuned model is the same size as the original.
  - [Resource intensive]{style="color: red;"}: full fine-tuning needs the full
    optimizer state, about 16 bytes per parameter (lecture 12).
  - [Storage demands]{style="color: red;"}: $n$ downstream tasks means $n$ copies of
    the model.

Solution: update a small *part* of the model rather than all of it.

- Low-rank adaptation of the weight updates: **LoRA**
- Train and prepend soft prompts: prompt tuning


## Full Finetuning Example

![Text classification performance on the [Stanford Natural Language Inference (SNLI) Corpus](https://nlp.stanford.edu/projects/snli/). Ordered pairs of sentences are classified by their logical relationship: either contradicted, entailed (implied), or neutral. Default fine-tuning parameters were used when not otherwise specified. <https://learn.microsoft.com/en-us/ai/playbook/technology-guidance/generative-ai/working-with-llms/fine-tuning>](figs/13-Training-and-Evaluating-LLMs/snli-finetuning-accuracy.png)

Hugging Face's tutorials are the fastest way to try this yourself:
[fine-tuning a pretrained model](https://huggingface.co/docs/transformers/training)
(BERT on Yelp reviews),
[text classification on GLUE](https://github.com/huggingface/notebooks/blob/main/examples/text_classification.ipynb)
and [question answering on SQuAD](https://github.com/huggingface/notebooks/blob/main/examples/question_answering.ipynb).


## Low Rank Adaptation

- Deploying independent instances of downstream fine-tuned models can be prohibitive
  (e.g. GPT-3, 175B params, 700GB\@fp32)
- Instead, freeze the pre-trained model and inject *trainable rank decomposition
  matrices* into each layer
- Reduce trainable parameters by 10,000x!!
- On-par or better than finetuning on RoBERTa, DeBERTa, GPT-2 and GPT-3


![](figs/13-Training-and-Evaluating-LLMs/lora-diagram.png)


*E. J. Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models." arXiv, Oct. 16, 2021. <http://arxiv.org/abs/2106.09685>*

## Low Rank Adaptation (cont.)

![](figs/13-Training-and-Evaluating-LLMs/lora-diagram.png)


- Aghajanyan et al show that pretrained language models have a low "intrinsic dimension"
- Updates to weight matrices likely have a low "intrinsic rank" during training
- Found that even very low rank (e.g. $r=1$ or 2) with GPT-3 175B is effective where
  full rank (embedding dimension) is 12,288


*E. J. Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models." arXiv, Oct. 16, 2021. <http://arxiv.org/abs/2106.09685>*\
*A. Aghajanyan et al., "Intrinsic Dimensionality Explains the Effectiveness of Language Model Fine-Tuning". arXiv:2012.13255 [cs], December 2020. URL <http://arxiv.org/abs/2012.13255>.*

## Reminder: Rank of a Matrix

- The number of linearly independent rows or columns of a matrix
- Determines the dimension of the vector space spanned by the column vectors
- A measure of "dimensionality"
- A $d \times k$ matrix of rank $r$ can be written as a product of a $d \times r$ and an
  $r \times k$ matrix: $r(d + k)$ numbers instead of $dk$

## LoRA: Method

Say you have pre-trained weights,

$$
W_0 \in \mathbb{R}^{d \times k}
$$

Represent update with a low rank decomposition

$$
W_0 + \Delta W = W_0 + BA\,,
$$

where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$ and the rank
$r \ll \min(d, k)$, is much less than the full rank.

For updates,

$$
h = (W_0 + \Delta W)x = W_0 x + \Delta W x = W_0 x + BAx
$$

Initialize $A$ to random Gaussian and $B$ to zero, so $\Delta W = 0$ at the start and
training begins from the pretrained model. $W_0$ is frozen; only $A$ and $B$ get
gradients.

- LoRA is a generalization of full finetuning: with $r = \min(d,k)$ it *is* full
  finetuning.
- The paper applies it only to the attention matrices $W_q$ and $W_v$; practice today
  often includes all linear layers.

*E. J. Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models." arXiv, Oct. 16, 2021. <http://arxiv.org/abs/2106.09685>*

## LoRA Results: GLUE (RoBERTa, DeBERTa)

| Model & Method | # Trainable Parameters | MNLI | SST-2 | MRPC | CoLA | QNLI | QQP | RTE | STS-B | Avg. |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| RoB~base~ (FT)\* | [125.0M]{.underline} | **87.6** | 94.8 | 90.2 | **63.6** | 92.8 | **91.9** | 78.7 | 91.2 | 86.4 |
| RoB~base~ (BitFit)\* | 0.1M | 84.7 | 93.7 | **92.7** | 62.0 | 91.8 | 84.0 | 81.5 | 90.8 | 85.2 |
| RoB~base~ (Adpt^D^)\* | 0.9M | 87.3~±.1~ | 94.7~±.3~ | 88.4~±.1~ | 62.6~±.9~ | 93.0~±.2~ | 90.6~±.0~ | 75.9~±2.2~ | 90.3~±.1~ | 85.4 |
| RoB~base~ (LoRA) | [0.3M]{.underline} | 87.5~±.3~ | **95.1**~±.2~ | 89.7~±.7~ | 63.4~±1.2~ | **93.3**~±.3~ | 90.8~±.1~ | **86.6**~±.7~ | **91.5**~±.2~ | **87.2** |
| RoB~large~ (FT)\* | [355.0M]{.underline} | 90.2 | **96.4** | **90.9** | 68.0 | 94.7 | **92.2** | 86.6 | 92.4 | 88.9 |
| RoB~large~ (Adpt^P^)† | 3.0M | 90.2~±.3~ | 96.1~±.3~ | 90.2~±.7~ | **68.3**~±1.0~ | **94.8**~±.2~ | **91.9**~±.1~ | 83.8~±2.9~ | 92.1~±.7~ | 88.4 |
| RoB~large~ (LoRA) | [0.8M]{.underline} | **90.6**~±.2~ | 96.2~±.5~ | **90.9**~±1.2~ | **68.2**~±1.9~ | **94.9**~±.3~ | 91.6~±.1~ | **87.4**~±2.5~ | **92.6**~±.2~ | **89.0** |
| DeB~XXL~ (FT)\* | [1500.0M]{.underline} | 91.8 | **97.2** | 92.0 | 72.0 | **96.0** | 92.7 | 93.9 | 92.9 | 91.1 |
| DeB~XXL~ (LoRA) | [4.7M]{.underline} | **91.9**~±.2~ | 96.9~±.2~ | **92.6**~±.6~ | **72.4**~±1.1~ | **96.0**~±.1~ | **92.9**~±.1~ | **94.9**~±.4~ | **93.0**~±.2~ | **91.3** |

- FT = full fine-tune; models are [RoBERTa](https://arxiv.org/abs/1907.11692) {base, large} and [DeBERTa](https://arxiv.org/abs/2006.03654) XXL
- GLUE benchmark – measure across 9 language tasks
- BitFit – train only the bias vectors; Adpt – inserts adapter layers between self-attention and MLP modules
- \* numbers published in prior work; † runs configured as in Houlsby et al. (2019) for a fair comparison. Selected rows of Table 2 in Hu et al.

*E. J. Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models." arXiv, Oct. 16, 2021. <http://arxiv.org/abs/2106.09685>*


## LoRA Results: E2E NLG (GPT-2 medium and large)

| Model & Method | # Trainable Parameters | BLEU | NIST | MET | ROUGE-L | CIDEr |
|---|---:|---:|---:|---:|---:|---:|
| GPT-2 M (FT)\* | [354.92M]{.underline} | 68.2 | 8.62 | 46.2 | 71.0 | 2.47 |
| GPT-2 M (Adapter^L^)\* | 0.37M | 66.3 | 8.41 | 45.0 | 69.8 | 2.40 |
| GPT-2 M (Adapter^L^)\* | 11.09M | 68.9 | 8.71 | 46.1 | 71.3 | 2.47 |
| GPT-2 M (FT^Top2^)\* | 25.19M | 68.1 | 8.59 | 46.0 | 70.8 | 2.41 |
| GPT-2 M (PreLayer)\* | 0.35M | 69.7 | 8.81 | 46.1 | 71.4 | 2.49 |
| GPT-2 M (LoRA) | [0.35M]{.underline} | **70.4**~±.1~ | **8.85**~±.02~ | **46.8**~±.2~ | **71.8**~±.1~ | **2.53**~±.02~ |
| GPT-2 L (FT)\* | [774.03M]{.underline} | 68.5 | 8.78 | 46.0 | 69.9 | 2.45 |
| GPT-2 L (Adapter^L^) | 0.88M | 69.1~±.1~ | 8.68~±.03~ | 46.3~±.0~ | 71.4~±.2~ | **2.49**~±.0~ |
| GPT-2 L (Adapter^L^) | 23.00M | 68.9~±.3~ | 8.70~±.04~ | 46.1~±.1~ | 71.3~±.2~ | 2.45~±.02~ |
| GPT-2 L (PreLayer)\* | 0.77M | 70.3 | 8.85 | 46.2 | 71.7 | 2.47 |
| GPT-2 L (LoRA) | [0.77M]{.underline} | **70.4**~±.1~ | **8.89**~±.02~ | **46.8**~±.2~ | **72.0**~±.2~ | 2.47~±.02~ |

GPT-2 medium (M) and large (L) with different adaptation methods on the E2E NLG
Challenge. For all metrics, higher is better. LoRA outperforms several baselines with
comparable or fewer trainable parameters. Confidence intervals are shown for experiments
we ran. \* indicates numbers published in prior works. FT = full fine-tune. Selected
rows of Table 3 in Hu et al.

*E. J. Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models." arXiv, Oct. 16, 2021. <http://arxiv.org/abs/2106.09685>*

## 1) Which weight matrices to target?

Given a parameter budget, which subset of weight matrices in a pre-trained Transformer
should we adapt? \# of trainable parameters = 18M (GPT-3 175B)

| Weight Type | $W_q$ | $W_k$ | $W_v$ | $W_o$ | $W_q, W_k$ | $W_q, W_v$ | $W_q, W_k, W_v, W_o$ |
|---|---:|---:|---:|---:|---:|---:|---:|
| Rank $r$ | 8 | 8 | 8 | 8 | 4 | 4 | 2 |
| WikiSQL ($\pm 0.5\%$) | 70.4 | 70.0 | 73.0 | 73.2 | 71.4 | **73.7** | **73.7** |
| MultiNLI ($\pm 0.1\%$) | 91.0 | 90.8 | 91.0 | 91.3 | 91.3 | 91.3 | **91.7** |

Validation accuracy on WikiSQL and MultiNLI after applying LoRA to different types of
attention weights in GPT-3, given the same number of trainable parameters. Adapting both
$W_q$ and $W_v$ gives the best performance overall. We find the standard deviation across
random seeds to be consistent for a given dataset, which we report in the first column.

[Rank 4 on two matrices ($W_q, W_v$), or even rank 2 on all four, is sufficient:
spreading a fixed budget over more matrices beats a higher rank on one.]{.mark}

*E. J. Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models." arXiv, Oct. 16, 2021. <http://arxiv.org/abs/2106.09685>*

## 2) What is the optimal rank?

| | Weight Type | $r=1$ | $r=2$ | $r=4$ | $r=8$ | $r=64$ |
|---|---|---:|---:|---:|---:|---:|
| WikiSQL ($\pm 0.5\%$) | $W_q$ | 68.8 | 69.6 | 70.5 | 70.4 | 70.0 |
| | $W_q, W_v$ | 73.4 | 73.3 | 73.7 | 73.8 | 73.5 |
| | $W_q, W_k, W_v, W_o$ | 74.1 | 73.7 | 74.0 | 74.0 | 73.9 |
| MultiNLI ($\pm 0.1\%$) | $W_q$ | 90.7 | 90.9 | 91.1 | 90.7 | 90.7 |
| | $W_q, W_v$ | 91.3 | 91.4 | 91.3 | 91.6 | 91.4 |
| | $W_q, W_k, W_v, W_o$ | 91.2 | 91.7 | 91.7 | 91.5 | 91.4 |

"Validation accuracy on WikiSQL and MultiNLI with different rank $r$. To our surprise, a
rank as small as one suffices for adapting both $W_q$ and $W_v$ on these datasets while
training $W_q$ alone needs a larger $r$."

*E. J. Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models." arXiv, Oct. 16, 2021. <http://arxiv.org/abs/2106.09685>*

## LoRA in practice (for the final project)

**Parameter count.** A LoRA on a $d \times k$ matrix adds $r(d + k)$ parameters.

*Worked example.* One $4096 \times 4096$ projection with $r = 8$:
$8 \times (4096 + 4096) = 65{,}536$ trainable parameters in place of
$4096^2 \approx 16.8\text{M}$, i.e. 0.4%. Over 32 layers and the four attention
projections that is about 8M parameters for a 7B–8B model (the exact count depends on
GQA, which shrinks $W_k$ and $W_v$).

**Scaling.** Implementations compute $h = W_0 x + \frac{\alpha}{r} BAx$; $\alpha$ is
a fixed constant, so the update's scale does not change when you change $r$.

**At inference** merge once, $W = W_0 + BA$: no extra latency, and the adapter
itself is a few MB you can swap per task.

**QLoRA** ([Dettmers et al., 2023](https://arxiv.org/abs/2305.14314)) keeps $W_0$ in
4-bit and trains the adapters in bf16: a 65B model fine-tunes on one 48 GB GPU.


With Hugging Face [PEFT](https://github.com/huggingface/peft):

```python
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,                     # rank
    lora_alpha=16,           # scale alpha / r
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj",
                    "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, config)
model.print_trainable_parameters()
# trainable params: ~8M || all params: ~8B || 0.1%
```

Checklist: start with $r = 8$ to 16 on all attention projections; evaluate on your
held-out set before and after; if the gain is small, retrieval or a better prompt
may be the cheaper fix.


The worked example as a calculator. Change `r`, or set `d_kv = 1024` for the
grouped-query $W_k$ and $W_v$ of Llama 3 8B (8 KV heads of dimension 128), and
watch the count move.

In [ ]:
d_model = 4096                 # hidden size: W_q and W_o are d_model x d_model
d_kv = 4096                    # output size of W_k and W_v (1024 with GQA in Llama 3 8B)
r = 8                          # LoRA rank
n_layers = 32
total_params = 8.03e9          # base model, for the percentage

shapes = {"q": (d_model, d_model), "k": (d_model, d_kv),
          "v": (d_model, d_kv), "o": (d_model, d_model)}

per_matrix = r * (d_model + d_model)
print(f"one {d_model} x {d_model} matrix: LoRA {per_matrix:,} vs full {d_model**2:,} "
      f"({100 * per_matrix / d_model**2:.2f}%)")

lora = n_layers * sum(r * (d + k) for d, k in shapes.values())
full = n_layers * sum(d * k for d, k in shapes.values())
print(f"{n_layers} layers x {len(shapes)} projections: LoRA {lora / 1e6:.2f}M trainable "
      f"vs {full / 1e6:,.0f}M in those matrices; {100 * lora / total_params:.2f}% of the model")
print(f"adapter size in bf16: {2 * lora / 2**20:.0f} MiB")

# Summary

One line per idea; the recap deck uses these.


## Summary

- A chat model is built in stages: pretraining on the web, supervised fine-tuning on demonstrations, a reward model from comparisons, RL against it (lecture 14).
- Pretraining data went from 0.5T tokens (GPT-3) to 15T (Llama 3); the targets are the inputs shifted by one, most data is seen about once, and quality filtering matters as much as quantity (lecture 12).
- Classifier fine-tuning replaces the output head and trains with cross-entropy; the last few blocks carry most of the adaptation.
- Instruction fine-tuning is the same next-token loss on templated (instruction, response) pairs; Alpaca used 52k generated examples, LIMA showed 1k curated ones can suffice.
- A reward model $r_\theta(x,y)$ is trained on comparisons with the Bradley–Terry loss $-\log\sigma(r_\theta(x,y_w) - r_\theta(x,y_l))$; it turns whole-response judgement into a training signal.
- Validation loss can improve while humans judge outputs worse: likelihood is a proxy for behaviour.
- A benchmark is a task, items and a scoring rule; its score is meaningful only if the items are held out, which for an LLM means decontaminated against the web (8-gram overlap, private splits, refreshed items).
- LLM judges agree with humans about as well as humans agree with each other but have position, verbosity and self-preference biases; swap order, give a rubric or reference, use another model family, check against humans.
- Pairwise grading ranks systems; a binary rubric tracks one; never an unanchored 1–10.
- Arena scores are Elo-scaled Bradley–Terry ratings, $\mathrm{Pr}(i \succ j) = 1/(1+10^{-(s_i-s_j)/400})$; a 100-point lead is a 64% win rate, and leaderboards that accept private submissions are gamed (Leaderboard Illusion).
- Task-specific evals stop memorization by verifiable outcomes (SWE-bench), un-googleable expertise (GPQA), adversarial collection with a private split (HLE), or novelty by construction (ARC-AGI).
- Build your own eval: narrow task, real items, dev/test split, verifiable grader first, agreement check ($\kappa$), and a standard error $\sqrt{\hat p(1-\hat p)/n}$ next to every score.
- RAG retrieves chunks by embedding distance (squared L2, inner product or cosine) and puts them in the prompt; rerank with a cross-encoder; evaluate with the RAG triad; context engineering generalizes it.
- LoRA freezes $W_0$ and trains $\Delta W = BA$ of rank $r$, adding $r(d+k)$ parameters per matrix; rank 1 to 8 on the attention projections matches full fine-tuning on GLUE and E2E; merge at inference.

## To Dive Deeper

- [Ouyang et al., *Training language models to follow instructions with human feedback* (2022)](https://arxiv.org/abs/2203.02155): the InstructGPT recipe (SFT, reward model, PPO); lecture 14 derives it.
- [Zheng et al., *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena* (2023)](https://arxiv.org/abs/2306.05685) and [Singh et al., *The Leaderboard Illusion* (2025)](https://arxiv.org/abs/2504.20879): read both before trusting a leaderboard.
- [Gao et al., *Retrieval-Augmented Generation for Large Language Models: A Survey* (2023)](https://arxiv.org/abs/2312.10997).
- [Hu et al., *LoRA* (2021)](https://arxiv.org/abs/2106.09685) and [Dettmers et al., *QLoRA* (2023)](https://arxiv.org/abs/2305.14314).
- From Sebastian Raschka: [LLM Training: RLHF and its Alternatives](https://magazine.sebastianraschka.com/p/llm-training-rlhf-and-its-alternatives),
  the LLMs-from-Scratch book and [repo](https://github.com/rasbt/LLMs-from-scratch), and
  [Understanding Reasoning LLMs](https://magazine.sebastianraschka.com/p/understanding-reasoning-llms).